In [1]:
import requests
import json
import time
import os

In [2]:
#Busqueda del ID del Equipo

API_KEY = "a5b99aab5ba5ae64d6f42eaa1b4f5628"
URL = "https://v3.football.api-sports.io/teams"
headers = {'x-rapidapi-key': API_KEY}

# Buscamos por nombre
params = {'search': 'Millonarios'}

print("Buscando ID de Millonarios...")
response = requests.get(URL, headers=headers, params=params).json()

if response.get('response'):
    print(f"{'ID':<10} | {'Nombre':<25} | {'País':<15}")
    print("-" * 55)
    for item in response['response']:
        t = item['team']
        print(f"{t['id']:<10} | {t['name']:<25} | {t['country']:<15}")
else:
    print("No se encontraron resultados. Verifica tu API Key.")

Buscando ID de Millonarios...
ID         | Nombre                    | País           
-------------------------------------------------------
1125       | Millonarios               | Colombia       
11057      | Millonarios U20           | Colombia       
15729      | Millonarios W             | Colombia       
25012      | Millonarios               | Bolivia        


In [13]:
# Configuración
API_KEY = "a5b99aab5ba5ae64d6f42eaa1b4f5628"
TEAM_ID = 1125
SEASON = 2025
URL_BASE = "https://v3.football.api-sports.io"

In [4]:
headers = {
    'x-rapidapi-host': "v3.football.api-sports.io",
    'x-rapidapi-key': API_KEY
}

In [14]:
def season_detailed_data():
    folder_name = f"Millonarios_{SEASON}_Stats_Detalladas"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    # 1. Obtener lista de partidos finalizados
    try:
        url_fixtures = f"{URL_BASE}/fixtures"
        params_fix = {'team': TEAM_ID, 'season': SEASON, 'status': 'FT'}
        response = requests.get(url_fixtures, headers=headers, params=params_fix)
        partidos = response.json().get('response', [])
    except Exception as e:
        print(f"Error al conectar con la API: {e}")
        return

    total = len(partidos)
    print(f"Se encontraron {total} partidos para procesar.\n")

    for i, match in enumerate(partidos, 1):
        # Datos para identificar el partido y nombrar el archivo
        fecha = match['fixture']['date'].split('T')[0]
        es_local = match['teams']['home']['id'] == TEAM_ID
        condicion = "Local" if es_local else "Visitante"
        rival = (match['teams']['away']['name'] if es_local else match['teams']['home']['name']).replace(" ", "_")
        liga = match['league']['name'].replace(" ", "_")
        
        file_name = f"{folder_name}/{fecha}_{condicion}_{rival}.json"

        # Saltamos si ya tenemos el archivo
        if os.path.exists(file_name):
            print(f"[{i}/{total}] Saltando: {fecha} vs {rival} (Ya descargado)")
            continue

        print(f"[{i}/{total}] Procesando: {fecha} vs {rival} ({condicion})...")

        # 2. Petición de estadísticas detalladas
        try:
            fixture_id = match['fixture']['id']
            res_stats = requests.get(f"{URL_BASE}/fixtures/players", headers=headers, params={'fixture': fixture_id, 'team': TEAM_ID})
            data_stats = res_stats.json()

            players_data = []
            if data_stats.get('response') and len(data_stats['response']) > 0:
                for p in data_stats['response'][0]['players']:
                    s = p['statistics'][0]
                    
                    # Tu bloque de estadísticas detalladas
                    stats_detalle = {
                        "nombre": p['player']['name'],
                        "posicion": s['games']['position'],
                        "minutos": s['games']['minutes'] or 0,
                        "calificacion": s['games']['rating'],
                        "titular": not s['games']['substitute'],
                        "rendimiento": {
                            "remates_totales": s['shots']['total'] or 0,
                            "remates_al_arco": s['shots']['on'] or 0,
                            "goles": s['goals']['total'] or 0,
                            "asistencias": s['goals']['assists'] or 0,
                            "pases": {
                                "totales": s['passes']['total'] or 0,
                                "precision": f"{s['passes']['accuracy']}%" if s['passes']['accuracy'] else "0%"
                            },
                            "defensa": {
                                "entradas": s['tackles']['total'] or 0,
                                "intercepciones": s['tackles']['interceptions'] or 0,
                                "despejes": s['tackles']['blocks'] or 0
                            },
                            "duelos": {
                                "totales": s['duels']['total'] or 0,
                                "ganados": s['duels']['won'] or 0
                            },
                            "faltas": {
                                "cometidas": s['fouls']['committed'] or 0,
                                "recibidas": s['fouls']['drawn'] or 0
                            },
                            "tarjetas": {
                                "amarilla": s['cards']['yellow'],
                                "roja": s['cards']['red']
                            }
                        }
                    }
                    players_data.append(stats_detalle)

            # 3. Guardar el archivo JSON
            resultado_final = {
                "metadata": {
                    "equipo": "Millonarios FC",
                    "rival": rival.replace("_", " "),
                    "condicion": condicion,
                    "campeonato": liga.replace("_", " "),
                    "fecha": fecha,
                    "resultado": f"{match['goals']['home']} - {match['goals']['away']}"
                },
                "jugadores": players_data
            }

            with open(file_name, 'w', encoding='utf-8') as f:
                json.dump(resultado_final, f, ensure_ascii=False, indent=4)
            print(f"   ✓ Datos guardados exitosamente.")

        except Exception as e:
            print(f"   ✗ Error en el partido {fecha}: {e}")

        # --- SLEEP OBLIGATORIO DE 30 SEGUNDOS ---
        if i < total:
            print("   Esperando 30 segundos para la próxima petición...", end="", flush=True)
            for _ in range(30):
                time.sleep(1)
                # Opcional: print(".", end="", flush=True)
            print(" ¡Listo!")

    print(f"\nProceso terminado. Revisa la carpeta: {folder_name}")

In [15]:
season_detailed_data()

Se encontraron 0 partidos para procesar.


Proceso terminado. Revisa la carpeta: Millonarios_2025_Stats_Detalladas


In [18]:
TEAM_ID = 1125      # Millonarios FC
LEAGUE_ID = 239   # ID específico de la liga (Ej: 241 para 2021)
SEASON_YEAR = 2025 # El año calendario

def download_by_league_id():
    folder_name = f"Millonarios_Liga_{LEAGUE_ID}_{SEASON_YEAR}"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    # 1. Obtener partidos de Millonarios en ESA liga específica
    # Usamos league y season para filtrar la tabla exacta
    params_fix = {
        'team': TEAM_ID, 
        'league': LEAGUE_ID, 
        'season': SEASON_YEAR, 
        #'status': 'FT'
    }
    
    try:
        response = requests.get(f"{URL_BASE}/fixtures", headers=headers, params=params_fix)
        partidos = response.json().get('response', [])
    except Exception as e:
        print(f"Error de conexión: {e}")
        return

    total = len(partidos)
    print(f"Se encontraron {total} partidos en la liga {LEAGUE_ID} ({SEASON_YEAR}).")

    for i, match in enumerate(partidos, 1):
        f_id = match['fixture']['id']
        fecha = match['fixture']['date'].split('T')[0]
        es_local = match['teams']['home']['id'] == TEAM_ID
        condicion = "Local" if es_local else "Visitante"
        rival = (match['teams']['away']['name'] if es_local else match['teams']['home']['name']).replace(" ", "_")
        
        file_name = f"{folder_name}/{fecha}_{condicion}_{rival}.json"

        if os.path.exists(file_name):
            print(f"[{i}/{total}] Saltando {fecha} (Ya existe).")
            continue

        print(f"[{i}/{total}] Procesando: {fecha} vs {rival}...")

        # 2. Intentar obtener estadísticas detalladas
        players_data = []
        try:
            res_stats = requests.get(f"{URL_BASE}/fixtures/players", headers=headers, params={'fixture': f_id, 'team': TEAM_ID})
            data_stats = res_stats.json()

            if data_stats.get('response') and len(data_stats['response']) > 0:
                for p in data_stats['response'][0]['players']:
                    s = p['statistics'][0]
                    # Aquí va tu bloque detallado (resumido aquí para brevedad)
                    players_data.append({
                        "nombre": p['player']['name'],
                        "posicion": s['games']['position'],
                        "minutos": s['games']['minutes'] or 0,
                        "calificacion": s['games']['rating'],
                        "titular": not s['games']['substitute'],
                        "rendimiento": {
                            "goles": s['goals']['total'] or 0,
                            "asistencias": s['goals']['assists'] or 0,
                            "pases": {"totales": s['passes']['total'] or 0, "precision": s['passes']['accuracy']}
                            # ... (puedes añadir el resto de campos que definimos antes)
                        }
                    })
            else:
                print(f"   ! Sin detalle de jugadores para el ID {f_id}")

        except Exception as e:
            print(f"   ✗ Error en estadísticas: {e}")

        # 3. Guardar el JSON (Incluso si players_data está vacío)
        resultado = {
            "metadata": {
                "fecha": fecha,
                "rival": rival.replace("_", " "),
                "condicion": condicion,
                "resultado": f"{match['goals']['home']} - {match['goals']['away']}",
                "fixture_id": f_id
            },
            "jugadores": players_data
        }

        with open(file_name, 'w', encoding='utf-8') as f:
            json.dump(resultado, f, ensure_ascii=False, indent=4)

        # 4. SLEEP OBLIGATORIO DE 30 SEGUNDOS
        if i < total:
            print(f"   --> Esperando 30 segundos...")
            time.sleep(30)

    print(f"\nDescarga finalizada en la carpeta {folder_name}")

In [19]:
download_by_league_id()

Se encontraron 0 partidos en la liga 239 (2025).

Descarga finalizada en la carpeta Millonarios_Liga_239_2025
